# Agent 1 - Single T-Test Tool

Goal: build the simplest statistical tool-calling agent.

The live Microsoft Agent Framework pattern is to pass Python functions through `tools=` or decorate with `@tool`. This notebook keeps a deterministic local function first, then shows optional Agent Framework wiring.

```mermaid
flowchart LR
    Prompt[Student prompt with data] --> Agent[TTestCoach]
    Agent --> Tool[Welch t-test tool]
    Tool --> Agent
    Agent --> Answer[Assumptions, p-value, interpretation]
```

## Coding Agent Prompt

```text
Build a single-tool statistical agent. The agent should ask for two numeric groups, call a Welch t-test tool, report sample sizes, means, p-value, and explain that the result is association evidence only.
```

In [ ]:
from typing import Annotated
import pandas as pd
from scipy import stats

def format_p_value(p: float) -> str:
    return f"{p:.4g}"

In [ ]:
def welch_t_test(group_a: list[float], group_b: list[float]) -> dict:
    """Run Welch's t-test for two independent samples."""
    result = stats.ttest_ind(group_a, group_b, equal_var=False)
    return {
        "test": "Welch t-test",
        "n_a": len(group_a),
        "n_b": len(group_b),
        "mean_a": sum(group_a) / len(group_a),
        "mean_b": sum(group_b) / len(group_b),
        "p_value": result.pvalue,
        "interpretation": "Small p-values are evidence against equal means, not proof of causality.",
    }

welch_t_test([1.2, 1.5, 1.7, 1.4], [0.8, 1.0, 1.1, 0.9])

In [ ]:
# Optional live-agent sketch.
# Requires `pip install agent-framework` and model credentials.
#
# from agent_framework import tool
#
# @tool(name="welch_t_test_tool", description="Run Welch's t-test on two numeric groups.")
# def welch_t_test_tool(group_a: list[float], group_b: list[float]) -> dict:
#     return welch_t_test(group_a, group_b)
#
# agent = chat_client.as_agent(
#     name="TTestCoach",
#     instructions="Help students structure and interpret Welch t-tests. Avoid causal claims.",
#     tools=[welch_t_test_tool],
# )
# result = await agent.run("Compare group A [1.2,1.5,1.7] and group B [0.8,1.0,1.1].")